# Traffic Control System — RREF Solution

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import FancyArrowPatch
from sympy import Matrix
import ipywidgets as widgets
from IPython.display import display

## Junction Diagram

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
ax.set_xlim(0, 10)
ax.set_ylim(0, 8)
ax.axis('off')

# Junction positions
junctions = {
    'A': (2, 6),
    'B': (8, 6),
    'C': (8, 2),
    'D': (2, 2)
}

# Draw junction circles
for name, (x, y) in junctions.items():
    circle = plt.Circle((x, y), 0.4, color='black', fill=False, linewidth=2)
    ax.add_patch(circle)
    ax.text(x, y, name, ha='center', va='center', fontsize=13, fontweight='bold')

# Arrow helper
def draw_arrow(ax, start, end, label, label_offset=(0, 0.3)):
    sx, sy = start
    ex, ey = end
    dx, dy = ex - sx, ey - sy
    length = (dx**2 + dy**2) ** 0.5
    ux, uy = dx / length, dy / length
    start_pt = (sx + 0.45 * ux, sy + 0.45 * uy)
    end_pt   = (ex - 0.45 * ux, ey - 0.45 * uy)
    ax.annotate("",
        xy=end_pt, xytext=start_pt,
        arrowprops=dict(arrowstyle='->', color='black', lw=1.8)
    )
    mx = (sx + ex) / 2 + label_offset[0]
    my = (sy + ey) / 2 + label_offset[1]
    ax.text(mx, my, label, ha='center', va='center', fontsize=11, color='black')

# Road segments
draw_arrow(ax, junctions['A'], junctions['B'], '$x_1$',  label_offset=(0,  0.35))
draw_arrow(ax, junctions['B'], junctions['C'], '$x_2$',  label_offset=(0.4, 0))
draw_arrow(ax, junctions['C'], junctions['D'], '$x_3$',  label_offset=(0, -0.35))
draw_arrow(ax, junctions['D'], junctions['A'], '$x_4$',  label_offset=(-0.4, 0))
draw_arrow(ax, junctions['B'], junctions['D'], '$x_5$',  label_offset=(0.4, 0))

# External flows
ax.annotate("", xy=junctions['A'], xytext=(0.5, 6),
    arrowprops=dict(arrowstyle='->', color='gray', lw=1.5, linestyle='dashed'))
ax.text(0.2, 6.3, '80 in', fontsize=9, color='gray')

ax.annotate("", xy=(9.5, 6), xytext=junctions['B'],
    arrowprops=dict(arrowstyle='->', color='gray', lw=1.5, linestyle='dashed'))
ax.text(9.6, 6.3, '30 out', fontsize=9, color='gray')

ax.annotate("", xy=junctions['C'], xytext=(9.5, 2),
    arrowprops=dict(arrowstyle='->', color='gray', lw=1.5, linestyle='dashed'))
ax.text(9.6, 2.3, '50 in', fontsize=9, color='gray')

ax.annotate("", xy=(0.5, 2), xytext=junctions['D'],
    arrowprops=dict(arrowstyle='->', color='gray', lw=1.5, linestyle='dashed'))
ax.text(0.1, 1.7, '60 out', fontsize=9, color='gray')

ax.set_title('Traffic Network — Junction Diagram', fontsize=13, pad=12)
plt.tight_layout()
plt.show()

## Step 1 — Augmented Matrix

In [ ]:
# Junction A:  x1 + x2              = 80
# Junction B:       x2 - x3 + x4   = 30
# Junction C:                x4 + x5 = 50
# Junction D:  x1              + x5 = 60

A = np.array([
    [1, 1,  0, 0, 0],
    [0, 1, -1, 1, 0],
    [0, 0,  0, 1, 1],
    [1, 0,  0, 0, 1]
], dtype=float)

b = np.array([80, 30, 50, 60], dtype=float)

augmented = np.column_stack((A, b))

print("Augmented Matrix [A | b]:")
print(augmented)

## Step 2 — Manual Row Reduction

In [ ]:
M = augmented.copy()

M[3] = M[3] - M[0]      # R4 = R4 - R1  (eliminate x1 from row 4)
M[0] = M[0] - M[1]      # R1 = R1 - R2  (eliminate x2 from row 1)
M[3] = M[3] + M[1]      # R4 = R4 + R2  (eliminate x2 from row 4)
M[[2, 3]] = M[[3, 2]]   # swap R3 and R4 (bring pivot into position)
M[2] = -M[2]            # R3 = -R3      (make pivot positive)
M[0] = M[0] - M[2]      # R1 = R1 - R3  (eliminate x3 from row 1)
M[1] = M[1] + M[2]      # R2 = R2 + R3  (eliminate x3 from row 2)
M[2] = M[2] + M[3]      # R3 = R3 + R4  (eliminate x4 from row 3)

print("RREF (manual):")
print(M)

## Step 3 — Verification with SymPy

In [ ]:
M_sympy = Matrix([
    [1, 1,  0, 0, 0, 80],
    [0, 1, -1, 1, 0, 30],
    [0, 0,  0, 1, 1, 50],
    [1, 0,  0, 0, 1, 60]
])

rref_matrix, pivot_cols = M_sympy.rref()

print("SymPy RREF:")
print(rref_matrix)
print()
print("Pivot columns:", pivot_cols)
print("x1, x2, x3, x4 are basic variables (have pivots)")
print("x5 is the free variable (no pivot in column 4)")

## Step 4 — General Solution

In [ ]:
# Reading off the RREF row by row:
#
#   Row 1:  x1 + x5 = 60   =>   x1 = 60 - t
#   Row 2:  x2 - x5 = 20   =>   x2 = 20 + t
#   Row 3:  x3      = 40   =>   x3 = 40  (fixed for all t)
#   Row 4:  x4 + x5 = 50   =>   x4 = 50 - t
#   x5 = t  (free variable, physically valid range: 0 <= t <= 50)

print("General solution (let x5 = t):")
print("  x1 = 60 - t")
print("  x2 = 20 + t")
print("  x3 = 40")
print("  x4 = 50 - t")
print("  x5 = t")
print()
print("Non-negativity constraints:")
print("  x1 >= 0  =>  t <= 60")
print("  x4 >= 0  =>  t <= 50   (binding constraint)")
print("  x5 >= 0  =>  t >= 0")
print()
print("Valid range:  0 <= t <= 50")

## Step 5 — Example and Verification

In [ ]:
for t in [0, 25, 40, 50]:
    X1 = 60 - t
    X2 = 20 + t
    X3 = 40
    X4 = 50 - t
    X5 = t

    eq1 = X1 + X2
    eq2 = X2 - X3 + X4
    eq3 = X4 + X5
    eq4 = X1 + X5

    ok = eq1 == 80 and eq2 == 30 and eq3 == 50 and eq4 == 60

    print(f"t = {t:2d}  =>  x1={X1:3}, x2={X2:3}, x3={X3}, x4={X4:3}, x5={X5:3}  |  checks pass: {ok}")

## Step 6 — Live Simulation (Interactive Slider)

In [ ]:
def run_simulation(t):
    # --- Compute all flows from the general solution ---
    x1 = 60 - t
    x2 = 20 + t
    x3 = 40
    x4 = 50 - t
    x5 = t

    # --- Verify all 4 junction equations ---
    checks = [
        ("Junction A  x1 + x2",          x1 + x2,          80),
        ("Junction B  x2 - x3 + x4",     x2 - x3 + x4,     30),
        ("Junction C  x4 + x5",          x4 + x5,          50),
        ("Junction D  x1 + x5",          x1 + x5,          60),
    ]

    # --- Draw the updated network diagram ---
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))

    # ---- LEFT: Junction network with live flow labels ----
    ax = axes[0]
    ax.set_xlim(0, 10)
    ax.set_ylim(0, 8)
    ax.axis('off')
    ax.set_title(f'Traffic Network   (t = x5 = {t})', fontsize=12, pad=10)

    junctions = {'A': (2, 6), 'B': (8, 6), 'C': (8, 2), 'D': (2, 2)}
    flows     = {'x1': x1,   'x2': x2,   'x3': x3,   'x4': x4,   'x5': x5}

    # Road colors
    colors = {'x1': '#2563EB', 'x2': '#16A34A', 'x3': '#DC2626', 'x4': '#7C3AED', 'x5': '#B45309'}

    def draw_sim_arrow(ax, start, end, flow_val, var_name, loffset=(0, 0.35)):
        sx, sy = start
        ex, ey = end
        dx, dy = ex - sx, ey - sy
        length = (dx**2 + dy**2) ** 0.5
        ux, uy = dx / length, dy / length
        sp = (sx + 0.45 * ux, sy + 0.45 * uy)
        ep = (ex - 0.45 * ux, ey - 0.45 * uy)
        # Line width scales with flow volume
        lw = max(1.0, min(6.0, 1.0 + flow_val / 15.0))
        color = colors[var_name]
        ax.annotate("", xy=ep, xytext=sp,
            arrowprops=dict(arrowstyle='->', color=color, lw=lw))
        mx = (sx + ex) / 2 + loffset[0]
        my = (sy + ey) / 2 + loffset[1]
        ax.text(mx, my, f'{var_name}={flow_val}', ha='center', va='center',
                fontsize=10, fontweight='bold', color=color,
                bbox=dict(boxstyle='round,pad=0.2', fc='white', ec=color, alpha=0.85))

    draw_sim_arrow(ax, junctions['A'], junctions['B'], x1, 'x1', loffset=(0,  0.45))
    draw_sim_arrow(ax, junctions['B'], junctions['C'], x2, 'x2', loffset=(0.6, 0))
    draw_sim_arrow(ax, junctions['C'], junctions['D'], x3, 'x3', loffset=(0, -0.45))
    draw_sim_arrow(ax, junctions['D'], junctions['A'], x4, 'x4', loffset=(-0.6, 0))
    draw_sim_arrow(ax, junctions['B'], junctions['D'], x5, 'x5', loffset=(0.6, 0))

    # Junction circles
    for name, (x, y) in junctions.items():
        circle = plt.Circle((x, y), 0.4, color='black', fill=False, linewidth=2)
        ax.add_patch(circle)
        ax.text(x, y, name, ha='center', va='center', fontsize=13, fontweight='bold')

    # External flows
    ax.annotate("", xy=junctions['A'], xytext=(0.5, 6),
        arrowprops=dict(arrowstyle='->', color='gray', lw=1.5, linestyle='dashed'))
    ax.text(0.2, 6.3, '80 in', fontsize=8, color='gray')
    ax.annotate("", xy=(9.5, 6), xytext=junctions['B'],
        arrowprops=dict(arrowstyle='->', color='gray', lw=1.5, linestyle='dashed'))
    ax.text(9.6, 6.3, '30 out', fontsize=8, color='gray')
    ax.annotate("", xy=junctions['C'], xytext=(9.5, 2),
        arrowprops=dict(arrowstyle='->', color='gray', lw=1.5, linestyle='dashed'))
    ax.text(9.6, 2.3, '50 in', fontsize=8, color='gray')
    ax.annotate("", xy=(0.5, 2), xytext=junctions['D'],
        arrowprops=dict(arrowstyle='->', color='gray', lw=1.5, linestyle='dashed'))
    ax.text(0.1, 1.7, '60 out', fontsize=8, color='gray')

    # ---- RIGHT: Bar chart of all flow values ----
    ax2 = axes[1]
    var_names  = ['x1', 'x2', 'x3', 'x4', 'x5']
    var_values = [x1,   x2,   x3,   x4,   x5]
    bar_colors = [colors[v] for v in var_names]

    bars = ax2.bar(var_names, var_values, color=bar_colors, width=0.5, edgecolor='white', linewidth=1.2)

    # Value labels on bars
    for bar, val in zip(bars, var_values):
        ax2.text(bar.get_x() + bar.get_width() / 2,
                 bar.get_height() + 0.8,
                 str(int(val)), ha='center', va='bottom', fontsize=11, fontweight='bold')

    ax2.set_ylim(0, 90)
    ax2.set_ylabel('Vehicles per hour', fontsize=10)
    ax2.set_title('Flow on each road', fontsize=12)
    ax2.axhline(y=0, color='black', linewidth=0.8)
    ax2.tick_params(axis='x', labelsize=11)

    # Highlight x5 as the free variable
    ax2.get_xticklabels()[4].set_fontweight('bold')
    ax2.text(4, var_values[4] + 5, 'free', ha='center', fontsize=8, color=colors['x5'])
    ax2.text(2, var_values[2] + 5, 'fixed', ha='center', fontsize=8, color=colors['x3'])

    # ---- Verification results printed below charts ----
    print(f"\nt = {t}  |  x1={x1}  x2={x2}  x3={x3}  x4={x4}  x5={x5}")
    print("-" * 55)
    all_ok = True
    for label, val, expected in checks:
        ok = val == expected
        if not ok:
            all_ok = False
        status = "PASS" if ok else "FAIL"
        print(f"  {label} = {val:3d}   (expected {expected})   {status}")
    print("-" * 55)
    print("  All junction equations satisfied:", all_ok)

    plt.tight_layout()
    plt.show()


# --- Slider widget ---
slider = widgets.IntSlider(
    value=25,
    min=0,
    max=50,
    step=1,
    description='x5 = t :',
    continuous_update=False,
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='500px')
)

widgets.interact(run_simulation, t=slider)